# Straight-through vs VIMCO on a phylogenetics problem

This notebook compares gradient estimators for the **tree topology** on an actual
likelihood-based phylogenetic posterior, with **jointly optimised branch lengths**.

The variational distribution factorises as `q(topology) q(branch lengths)`:

* **Topology**: a conditional clade distribution (CCD / subsplit Bayesian network)
  over rooted topologies -- discrete, so its gradient needs an estimator.
* **Branch lengths**: a root height plus one node-height **ratio per split**
  (indexed by clade), mapped to node heights by TreeFlow's node-height ratio
  transform -- continuous, so its gradient is pathwise (reparameterised).

The likelihood is Felsenstein's pruning under Jukes-Cantor. We compare topology
estimators:

* **Straight-through** -- the relaxed child gather
  (`clade_straight_through_log_likelihood`): the forward pass is the exact
  likelihood of the sampled tree, and the gradient flows to the clade model
  through the child selection (and to the branch lengths pathwise).
* **Score function + leave-one-out (RLOO)** -- REINFORCE on the topology, branch
  lengths pathwise.
* **VIMCO** -- the multi-sample importance-weighted bound over topologies.

For a small taxon set we can enumerate every topology, so we have an exact
posterior over topologies (given the variational branch lengths) to score against.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from treeflow import DEFAULT_FLOAT_DTYPE_TF as FLOAT
from treeflow.conditional_clade.support import ConditionalCladeSupport
from treeflow.conditional_clade.distribution import ConditionalCladeDistribution
from treeflow.conditional_clade.clade import full_clade
from treeflow.conditional_clade.clade_likelihood import clade_straight_through_log_likelihood
from treeflow.conditional_clade.estimators import score_function_surrogate, leave_one_out_baseline, vimco_surrogate
from treeflow.tree.topology.numpy_tree_topology import NumpyTreeTopology
from treeflow.tree.topology.tensorflow_tree_topology import numpy_topology_to_tensor
from treeflow.traversal.ratio_transform import ratios_to_node_heights

tf.random.set_seed(0)
np.set_printoptions(precision=3, suppress=True)
STATE = 4  # nucleotides; Jukes-Cantor

## 1. Substitution model and a simulated dataset

Jukes-Cantor transition matrix `P(t)`, and sequences simulated down a fixed true
tree so we know the answer.

In [ ]:
def jc69(t):
    e = tf.exp(-4.0 / 3.0 * t)
    same = 0.25 + 0.75 * e
    diff = 0.25 - 0.25 * e
    return diff * tf.ones((STATE, STATE), FLOAT) + (same - diff) * tf.eye(STATE, dtype=FLOAT)

N_TAXA = 5
support = ConditionalCladeSupport(N_TAXA, taxon_set=[f't{i}' for i in range(N_TAXA)])
FULL = full_clade(N_TAXA)
frequencies = tf.constant(np.ones(STATE) / STATE, FLOAT)

# A fixed 'true' tree (parent_indices) and node heights to simulate from.
true_parent = np.array([5, 5, 6, 7, 7, 6, 8, 8], dtype=np.int32)  # 5 taxa
true_topology = NumpyTreeTopology(parent_indices=true_parent)
true_child = true_topology.child_indices
true_height = {i: 0.0 for i in range(N_TAXA)}
true_height.update({5: 0.05, 6: 0.12, 7: 0.07, 8: 0.25})  # internal node heights

def simulate(sites, seed=0):
    rng = np.random.default_rng(seed)
    node_count = 2 * N_TAXA - 1
    states = np.zeros((node_count, sites), dtype=int)
    root = node_count - 1
    states[root] = rng.integers(0, STATE, size=sites)
    # preorder: parents before children
    for node in true_topology.preorder_indices:
        if node == root:
            continue
        parent = true_parent[node]
        t = true_height[parent] - true_height[node]
        P = jc69(tf.constant(t, FLOAT)).numpy()
        for s in range(sites):
            states[node, s] = rng.choice(STATE, p=P[states[parent, s]])
    leaf_states = states[:N_TAXA]  # [leaf, site]
    return tf.constant(np.eye(STATE)[leaf_states.T], FLOAT)  # [site, leaf, state]

sequences = simulate(sites=300, seed=1)
print('simulated alignment [sites, leaf, state]:', sequences.shape)

## 2. Variational model

`q(topology)` is a CCD parametrised by per-subsplit logits. `q(branch lengths)`
is a root height plus a node-height ratio per splittable clade; the ratio
transform turns these into node heights, hence branch lengths, hence per-edge
`P(t)`.

In [ ]:
topo_logits = tf.Variable(tf.zeros(support.subsplit_count, FLOAT))
ratio_logits = tf.Variable(tf.zeros(support.parent_clade_count, FLOAT))  # per splittable clade
root_raw = tf.Variable(tf.constant(-1.0, FLOAT))  # softplus -> root height
q = ConditionalCladeDistribution(support, topo_logits)
branch_variables = [ratio_logits, root_raw]

def clades_of_nodes(parent_indices):
    child = NumpyTreeTopology(parent_indices=parent_indices).child_indices
    clade = {i: 1 << i for i in range(N_TAXA)}
    for u in range(N_TAXA, 2 * N_TAXA - 1):
        clade[u] = clade[int(child[u][0])] | clade[int(child[u][1])]
    return clade

def branch_transition_fn(parent_indices):
    """Per-clade transition matrix from the variational branch lengths."""
    topology = numpy_topology_to_tensor(NumpyTreeTopology(parent_indices=parent_indices))
    clade = clades_of_nodes(parent_indices)
    root_height = tf.nn.softplus(root_raw)
    ratios = []
    for u in range(N_TAXA, 2 * N_TAXA - 1):
        if clade[u] == FULL:
            ratios.append(root_height)
        else:
            ratios.append(tf.sigmoid(ratio_logits[support.parent_clade_index[clade[u]]]))
    ratios = tf.stack(ratios)[tf.newaxis]
    heights = ratios_to_node_heights(topology, ratios, tf.zeros_like(ratios))[0]
    height = {u: heights[u - N_TAXA] for u in range(N_TAXA, 2 * N_TAXA - 1)}
    for i in range(N_TAXA):
        height[i] = tf.constant(0.0, FLOAT)
    branch = {}
    for node, c in clade.items():
        if c == FULL:
            continue
        branch[c] = height[int(parent_indices[node])] - height[node]
    default = tf.reduce_mean(tf.stack(list(branch.values())))
    return lambda c: jc69(branch.get(c, default))

def felsenstein_log_likelihood(parent_indices, transition_fn):
    """Plain Felsenstein on a fixed sampled tree (differentiable in branches only)."""
    clade = clades_of_nodes(parent_indices)
    child = NumpyTreeTopology(parent_indices=parent_indices).child_indices
    part = {i: sequences[..., i, :] for i in range(N_TAXA)}
    for u in range(N_TAXA, 2 * N_TAXA - 1):
        c0, c1 = int(child[u][0]), int(child[u][1])
        left = tf.linalg.matvec(transition_fn(clade[c0]), part[c0])
        right = tf.linalg.matvec(transition_fn(clade[c1]), part[c1])
        part[u] = left * right
    return tf.reduce_sum(frequencies * part[2 * N_TAXA - 2], axis=-1)  # per-site likelihood

print('variational parameters:', int(topo_logits.shape[0]), 'topology logits,',
      int(ratio_logits.shape[0]), 'ratios, 1 root height')

## 3. Exact posterior over topologies (reference)

For each topology, the marginal data log-likelihood at the current variational
branch lengths gives an (unnormalised) posterior over topologies to compare `q`
against. The true tree should accumulate most of the mass as inference proceeds.

In [ ]:
enumerated = support.enumerate_parent_indices()
true_flatset = None
def flatset(pi):
    a = support.parent_indices_to_assignment(pi)
    return tuple(sorted(support.assignment_flat_indices(a).tolist()))
true_flatset = flatset(true_parent)
true_index = [i for i, pi in enumerate(enumerated) if flatset(pi) == true_flatset][0]

def exact_topology_posterior():
    logliks = []
    for pi in enumerated:
        tfn = branch_transition_fn(pi)
        logliks.append(float(tf.reduce_sum(tf.math.log(felsenstein_log_likelihood(pi, tfn)))))
    logliks = np.array(logliks)
    return np.exp(logliks - logliks.max()) / np.exp(logliks - logliks.max()).sum()

def q_true_probability():
    probs = q.enumerate_probs().numpy()
    order = [flatset(pi) for pi in q.enumerate_parent_indices()]
    return float(probs[order.index(true_flatset)])

print('number of topologies:', len(enumerated), '| true tree index:', true_index)

## 4. Estimators

All maximise the same objective -- the expected data log-likelihood plus the
topology entropy (a tempered ELBO) -- and all optimise the branch lengths
pathwise; they differ only in the **topology** gradient.

In [ ]:
def sample_trees(batch_size, rng):
    return [q.sample_parent_indices(rng) for _ in range(batch_size)]

def grad_straight_through(rng, batch_size):
    trees = sample_trees(batch_size, rng)
    with tf.GradientTape() as tape:
        logliks = []
        for pi in trees:
            tfn = branch_transition_fn(pi)
            logliks.append(tf.reduce_sum(clade_straight_through_log_likelihood(
                q, pi, sequences, tfn, frequencies)))
        loss = -(tf.add_n(logliks) / batch_size + q.entropy())
    return tape.gradient(loss, [topo_logits] + branch_variables)

def _score_like(rng, batch_size):
    trees = sample_trees(batch_size, rng)
    flat = tf.constant([support.assignment_flat_indices(
        support.parent_indices_to_assignment(pi)) for pi in trees], tf.int32)
    log_q = q.log_prob_from_flat_indices(flat)  # [batch]
    logliks = tf.stack([
        tf.reduce_sum(tf.math.log(felsenstein_log_likelihood(pi, branch_transition_fn(pi))))
        for pi in trees])
    return log_q, logliks

def grad_score_loo(rng, batch_size):
    with tf.GradientTape() as tape:
        log_q, logliks = _score_like(rng, batch_size)
        baseline = leave_one_out_baseline(logliks)
        # maximise E_q[logL] + entropy: cost = -logL (we minimise)
        surrogate = score_function_surrogate(-logliks, log_q, -baseline) - q.entropy()
    return tape.gradient(surrogate, [topo_logits] + branch_variables)

def grad_vimco(rng, batch_size):
    with tf.GradientTape() as tape:
        log_q, logliks = _score_like(rng, batch_size)
        log_weights = logliks - log_q  # IWAE weights over topologies
        loss = -vimco_surrogate(log_q, log_weights)
    return tape.gradient(loss, [topo_logits] + branch_variables)

ESTIMATORS = {
    'straight-through': grad_straight_through,
    'score + leave-one-out': grad_score_loo,
    'VIMCO': grad_vimco,
}

## 5. Optimisation

We track the probability `q` places on the **true** topology, and how close `q`'s
topology marginal is to the exact posterior (total-variation distance).

In [ ]:
def reset_parameters(seed=0):
    rng = np.random.default_rng(seed)
    topo_logits.assign(tf.constant(0.1 * rng.standard_normal(support.subsplit_count), FLOAT))
    ratio_logits.assign(tf.zeros(support.parent_clade_count, FLOAT))
    root_raw.assign(tf.constant(-1.0, FLOAT))

def train(grad_fn, steps=400, lr=0.05, batch_size=8, seed=0, eval_every=10):
    reset_parameters(seed)
    opt = tf.optimizers.Adam(lr)
    rng = np.random.default_rng(100)
    history = []
    for step in tqdm(range(steps)):
        grads = grad_fn(rng, batch_size)
        opt.apply_gradients(zip(grads, [topo_logits] + branch_variables))
        if step % eval_every == 0:
            exact = exact_topology_posterior()
            qprobs = q.enumerate_probs().numpy()
            order = [flatset(pi) for pi in q.enumerate_parent_indices()]
            exact_aligned = np.array([exact[[flatset(pi) for pi in enumerated].index(o)] for o in order])
            tv = 0.5 * np.abs(qprobs - exact_aligned).sum()
            history.append((step, q_true_probability(), tv))
    return np.array(history)

histories = {}
for name, fn in ESTIMATORS.items():
    histories[name] = train(fn, steps=400, batch_size=8)
    print(f'{name:24s} q(true tree) = {histories[name][-1,1]:.3f}   TV to exact = {histories[name][-1,2]:.3f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, h in histories.items():
    ax[0].plot(h[:, 0], h[:, 1], label=name)
    ax[1].plot(h[:, 0], h[:, 2], label=name)
ax[0].axhline(exact_topology_posterior()[true_index], ls=':', color='k', label='exact posterior')
ax[0].set_xlabel('step'); ax[0].set_ylabel('q(true topology)'); ax[0].set_title('Recovering the true tree')
ax[1].set_xlabel('step'); ax[1].set_ylabel('TV(q topology, exact)'); ax[1].set_yscale('log')
ax[1].set_title('Distance to the exact topology posterior')
ax[0].legend(); ax[1].legend(); plt.tight_layout(); plt.show()

## 6. Summary

* The straight-through child-gather likelihood lets the **phylogenetic
  likelihood's gradient reach the clade model**, so topology and branch lengths
  are optimised jointly (branch lengths pathwise through the node-height ratio
  transform).
* On this likelihood-based posterior we compare the **straight-through** topology
  estimator against **score-function (RLOO)** and **VIMCO**, scoring each by how
  much probability `q` puts on the true topology and the total-variation distance
  to the exact (enumerated) topology posterior.
* Straight-through is biased but low variance; RLOO is unbiased; VIMCO ascends the
  multi-sample bound. The same machinery scales (with a restricted or
  embedding-based support and sampled candidate subsplits) beyond the enumerable
  regime used here for the exact reference.